# 📓 Notebook 3: LLM Simplification

**Purpose**: The core step — send each chunk to the local Ollama model with carefully crafted prompts, collect simplified output, and checkpoint progress.

**Features**:
- 🔄 **Auto-retry** — if output is too short (info likely dropped), re-prompts with a stricter instruction
- ⚡ **Parallel processing** — process multiple chunks at once (set `MAX_WORKERS` in config)
- 📊 **Progress bar** — tqdm shows real-time progress and ETA
- 💾 **Checkpoint resume** — crashes? Out of memory? Just re-run. Picks up where it left off.
- ✅ **Verification pass** — optional second LLM pass to flag info loss

**Inputs**: `chunks.json`, `glossary.json`, `book_summary.txt`  
**Output**: `simplified_chunks.json`

In [ ]:
# ── Imports & Configuration ─────────────────────────────────────────────────
import sys
import json
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import (
    CHUNKS_FILE, GLOSSARY_FILE, BOOK_SUMMARY_FILE,
    SIMPLIFIED_CHUNKS_FILE, CHECKPOINT_FILE,
    MODEL_NAME, OLLAMA_BASE_URL, TEMPERATURE,
    VERIFY_CHUNKS, MIN_ACCEPTABLE_RATIO, MAX_ACCEPTABLE_RATIO,
    MAX_RETRIES, MAX_WORKERS,
)
from src.llm_client import OllamaClient
from src.simplifier import simplify_chunks

In [ ]:
# ── Load Inputs ──────────────────────────────────────────────────────────────
with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)

with open(GLOSSARY_FILE, "r", encoding="utf-8") as f:
    glossary = json.load(f)

with open(BOOK_SUMMARY_FILE, "r", encoding="utf-8") as f:
    book_summary = f.read()

print(f"📦 Loaded: {len(chunks)} chunks, {len(glossary)} glossary terms, summary ({len(book_summary)} chars)")
print(f"\n⚙️  Settings:")
print(f"   Model:        {MODEL_NAME}")
print(f"   Workers:      {MAX_WORKERS} ({'parallel' if MAX_WORKERS > 1 else 'sequential + context'})")
print(f"   Retry:        up to {MAX_RETRIES}x if ratio < {MIN_ACCEPTABLE_RATIO} or > {MAX_ACCEPTABLE_RATIO}")
print(f"   Verification: {'ON' if VERIFY_CHUNKS else 'OFF'}")

In [ ]:
# ── Initialize LLM & Speed Check ────────────────────────────────────────────
llm = OllamaClient(model_name=MODEL_NAME, base_url=OLLAMA_BASE_URL)
assert llm.is_available(), f"❌ Ollama not reachable at {OLLAMA_BASE_URL}. Is it running?"

speed = llm.estimate_speed()
print(f"✅ Model: {speed['model_name']}")
print(f"⚡ Speed: {speed['tokens_per_second']:.1f} tokens/sec")
print(f"⏱️  Estimated time for a 200-page book: ~{speed['estimated_minutes_200pg']:.0f} minutes")

In [ ]:
# ── Run Simplification ───────────────────────────────────────────────────────
# This is the main event. Features:
# - Auto-retry if output is too short/long (ratio check)
# - Parallel processing (MAX_WORKERS > 1) or sequential with context (MAX_WORKERS = 1)
# - Progress bar with ETA
# - Checkpoint after every chunk — safe to interrupt and resume
# - Optional verification pass to flag info loss

output_chunks = simplify_chunks(
    chunks=chunks,
    llm=llm,
    book_summary=book_summary,
    glossary=glossary,
    temperature=TEMPERATURE,
    min_ratio=MIN_ACCEPTABLE_RATIO,
    max_ratio=MAX_ACCEPTABLE_RATIO,
    max_retries=MAX_RETRIES,
    max_workers=MAX_WORKERS,
    checkpoint_path=CHECKPOINT_FILE,
    verify=VERIFY_CHUNKS,
)

In [ ]:
# ── Save Simplified Chunks ───────────────────────────────────────────────────
with open(SIMPLIFIED_CHUNKS_FILE, "w", encoding="utf-8") as f:
    json.dump(output_chunks, f, ensure_ascii=False, indent=2)

print(f"💾 Saved → {SIMPLIFIED_CHUNKS_FILE.name}")
print(f"\n✅ Simplification complete. Proceed to 04_assemble.ipynb")

In [ ]:
# ── Preview: Before vs After ─────────────────────────────────────────────────
# Show a side-by-side preview of the first substantive chunk

preview_chunk = None
for c in output_chunks:
    if len(c['text'].split()) > 20:
        preview_chunk = c
        break

if preview_chunk:
    orig = preview_chunk['text']
    simp = preview_chunk['simplified_text']
    print(f"📖 Chunk {preview_chunk['chunk_id']} — {preview_chunk['chapter']}")
    print(f"   Original:   {len(orig.split())} words")
    print(f"   Simplified: {len(simp.split())} words")
    print(f"   Ratio:      {len(simp.split()) / max(len(orig.split()),1):.2f}x")
    print(f"\n{'─'*40} ORIGINAL {'─'*40}")
    print(orig[:500])
    print(f"\n{'─'*40} SIMPLIFIED {'─'*38}")
    print(simp[:500])